# Query Transformation — Hands-On

Offline demo of HyDE, multi-query expansion, RRF fusion, decomposition, and routing.

## 0. Setup

In [ ]:
%pip install -q numpy
import hashlib, numpy as np
corpus={"hyde":"HyDE embeds a hypothetical document to retrieve real documents.","mmr":"MMR selects relevant and diverse chunks for RAG.","route":"Query routing sends questions to the right data source.","step":"Step-back prompting retrieves broader principles before details."}
def embed(t, dim=40):
    v=np.zeros(dim)
    for tok in t.lower().split(): v[int(hashlib.md5(tok.encode()).hexdigest(),16)%dim]+=1
    return v/(np.linalg.norm(v)+1e-9)
vecs={k:embed(v) for k,v in corpus.items()}

## 1. Raw retrieval

In [ ]:
def retrieve(q, k=2):
    qv=embed(q); return sorted([(float(qv@v),d) for d,v in vecs.items()], reverse=True)[:k]
q="better search probes for rag"
print(retrieve(q,3))

## 2. HyDE probe

In [ ]:
def hyde(q): return f"A RAG technique that rewrites {q} into a detailed document about retrieval, queries, routing, and evidence."
print(hyde(q))
print(retrieve(hyde(q),3))

## 3. Multi-query + RRF

In [ ]:
def variants(q): return [q, "query routing and transformation", "HyDE multi query retrieval"]
def rrf(lists, k=60):
    scores={}
    for ranked in lists:
        for r, (_,d) in enumerate(ranked,1): scores[d]=scores.get(d,0)+1/(k+r)
    return sorted([(s,d) for d,s in scores.items()], reverse=True)
ranked=[retrieve(v,3) for v in variants(q)]
print(rrf(ranked))

## 4. Decomposition and routing

In [ ]:
def decompose(q): return ["what is query transformation", "how does routing choose a source"]
def route(q):
    if "source" in q or "routing" in q: return "router-index"
    if "hyde" in q.lower(): return "semantic-index"
    return "general-index"
for sub in decompose(q): print(sub, "->", route(sub), retrieve(sub,1))

## 5. Exercise prompts
1. Add a fallback route.
2. Penalize duplicate results.
3. Compare raw vs transformed recall for hand-labeled relevant docs.